# Data mentioned in the text of the Data extraction section
Every number cited in *Data extraction* (Tables 1–3), recomputed from `cultura_enriched_v3.2`. Tables themselves are drawn in `01_data_extraction`; catalog categories follow `notebooks_analysis/02_catalog_coverage`.

In [1]:
import tempfile
import duckdb
import polars as pl

DB_PATH = '../../data/cultura/cultura_enriched_v3.2.duckdb'
INTERNATIONAL_QID = 'Q1072012'
pl.Config.set_tbl_rows(40)

con = duckdb.connect(DB_PATH, read_only=True)
con.sql(f"SET memory_limit='4GB'; SET threads=6; SET temp_directory='{tempfile.mkdtemp()}';")
total = con.sql('SELECT COUNT(*) FROM individual').fetchone()[0]
pct = lambda n, d=total: round(100 * n / d, 1)
print('individuals:', f'{total:,}')

individuals: 13,002,897


## Field coverage (paragraph 2, Table 1)

In [2]:
coverage = con.sql('''
    SELECT
        COUNT(*) FILTER (WHERE len(individual.birth_date) > 0)             AS birth_date,
        COUNT(*) FILTER (WHERE len(individual.death_date) > 0)             AS death_date,
        COUNT(individual.place_of_birth.entity.qid)                        AS place_of_birth,
        COUNT(individual.place_of_death.entity.qid)                        AS place_of_death,
        COUNT(*) FILTER (WHERE len(individual.occupation) > 0)             AS occupation,
        COUNT(individual.sex_or_gender)                                    AS gender,
        COUNT(*) FILTER (WHERE len(individual.sitelink) > 0)               AS sitelink,
        COUNT(*) FILTER (WHERE len(individual.external_id) > 0)            AS external_id,
        COUNT(*) FILTER (WHERE len(individual.external_id) > 0 AND len(individual.sitelink) = 0) AS identifier_without_sitelink,
        COUNT(*) FILTER (WHERE len(individual.external_id) = 0 AND len(individual.sitelink) = 0) AS neither
    FROM individual
''').pl().transpose(include_header=True, header_name='field', column_names=['individuals'])
coverage.with_columns((pl.col('individuals') / total * 100).round(1).alias('% of individuals'))

field,individuals,% of individuals
str,i64,f64
"""birth_date""",7746331,59.6
"""death_date""",3899892,30.0
"""place_of_birth""",3724787,28.6
"""place_of_death""",1585483,12.2
"""occupation""",9038108,69.5
"""gender""",8965424,68.9
"""sitelink""",4959581,38.1
"""external_id""",8738235,67.2
"""identifier_without_sitelink""",4588045,35.3


## Places (paragraph 3)

In [3]:
places = con.sql('''
    SELECT COUNT(*) AS places, COUNT(place.coordinates) AS coordinates, COUNT(place.sitelink) AS wikipedia_url,
           COUNT(place.existence.inception) AS inception, COUNT(place.existence.dissolution) AS dissolution,
           COUNT(*) FILTER (WHERE place.is_settlement IS NOT NULL) AS settlement_flag_filled
    FROM place
''').pl()
n_places = places['places'].item()
print({c: f'{places[c].item():,} ({pct(places[c].item(), n_places)}%)' for c in places.columns})

{'places': '361,751 (100.0%)', 'coordinates': '346,267 (95.7%)', 'wikipedia_url': '255,502 (70.6%)', 'inception': '72,892 (20.1%)', 'dissolution': '13,530 (3.7%)', 'settlement_flag_filled': '0 (0.0%)'}


## Wikimedia links, identifiers and works (paragraph 4)

In [4]:
links = con.sql('''
    SELECT (SELECT COUNT(*) FROM individual_sitelink) AS sitelinks,
           (SELECT COUNT(DISTINCT site_url) FROM individual_sitelink) AS projects,
           (SELECT COUNT(*) FROM sitelink WHERE url LIKE '%.wikipedia.org') AS wikipedia_editions,
           (SELECT COUNT(*) FROM individual_identifier) AS identifier_records,
           (SELECT COUNT(DISTINCT pid) FROM individual_identifier) AS identifier_systems_used,
           (SELECT COUNT(*) FROM identifier) AS identifier_properties,
           (SELECT COUNT(identifier.issuer_country) FROM identifier) AS with_country,
           (SELECT COUNT(identifier.number_of_records) FROM identifier) AS with_number_of_records,
           (SELECT COUNT(*) FROM individual_work) AS person_work_links,
           (SELECT COUNT(*) FROM work) AS works,
           (SELECT COUNT(*) FILTER (WHERE work.publication_date IS NOT NULL OR work.inception IS NOT NULL) FROM work) AS works_dated,
           (SELECT COUNT(*) FILTER (WHERE work.publication_date IS NOT NULL) FROM work) AS works_published
''').pl()
row = links.row(0, named=True)
print({k: f'{v:,}' for k, v in row.items()})
print('identifier properties with country:', pct(row['with_country'], row['identifier_properties']), '%')
print('identifier properties with number of records:', pct(row['with_number_of_records'], row['identifier_properties']), '%')
print('works dated:', pct(row['works_dated'], row['works']), '%; with publication date:', pct(row['works_published'], row['works']), '%')

{'sitelinks': '15,544,183', 'projects': '607', 'wikipedia_editions': '347', 'identifier_records': '36,880,469', 'identifier_systems_used': '2,342', 'identifier_properties': '10,198', 'with_country': '6,097', 'with_number_of_records': '3,836', 'person_work_links': '38,487,019', 'works': '17,050,395', 'works_dated': '15,782,870', 'works_published': '14,857,895'}
identifier properties with country: 59.8 %
identifier properties with number of records: 37.6 %
works dated: 92.6 %; with publication date: 87.1 %


## Catalog categories (paragraph 4)
Same rule as `02_catalog_coverage`: Western / non-Western by the present-day state of the issuer country; International = issuer country Q1072012.

In [5]:
catalogs = con.sql(f'''
    WITH place_present_state AS (
        SELECT place.entity.qid,
               CASE WHEN place.present_day_state.name IS NOT NULL THEN place.present_day_state ELSE parent.present_day_state END AS state
        FROM place LEFT JOIN place AS parent ON parent.entity.qid = place.country.qid
    )
    SELECT CASE WHEN identifier.issuer_country.name = '{INTERNATIONAL_QID}' THEN 'international'
                WHEN place_present_state.state.is_western THEN 'western'
                WHEN NOT place_present_state.state.is_western THEN 'non_western' END AS category,
           COUNT(*) AS identifier_properties
    FROM identifier
    LEFT JOIN place_present_state ON place_present_state.qid = identifier.issuer_country.name
    GROUP BY 1 ORDER BY 2 DESC
''').pl()
catalogs.with_columns((pl.col('identifier_properties') / pl.col('identifier_properties').sum() * 100).round(1).alias('%'))

category,identifier_properties,%
str,i64,f64
"""western""",4417,43.3
null,4235,41.5
"""non_western""",1545,15.2
"""international""",1,0.0


## Unique values and gender categories (paragraph 5, Table 2)

In [6]:
GENDER_LABELS = {'Q6581097': 'male', 'Q6581072': 'female', 'Q1052281': 'trans woman', 'Q48270': 'non-binary', 'Q2449503': 'trans man'}
genders = con.sql('SELECT individual.sex_or_gender AS qid, COUNT(*) AS individuals FROM individual GROUP BY 1 ORDER BY 2 DESC').pl()
print('distinct gender values:', genders['qid'].drop_nulls().n_unique())
genders.head(6).with_columns(
    pl.col('qid').replace_strict(GENDER_LABELS, default=None).alias('label'),
    (pl.col('individuals') / total * 100).round(1).alias('% of individuals'),
)

distinct gender values: 44


qid,individuals,label,% of individuals
str,i64,str,f64
"""Q6581097""",6109794,"""male""",47.0
null,4037473,null,31.1
"""Q6581072""",2849324,"""female""",21.9
"""Q1052281""",2649,"""trans woman""",0.0
"""Q48270""",1538,"""non-binary""",0.0
"""Q2449503""",659,"""trans man""",0.0


## Notability (paragraph 6, Table 3)
The `notability` field is empty in v3.2 (enrichment step 08 not run), so edition counts are recomputed from `individual_sitelink` with the rule of `08_individual_notability.py`.

In [7]:
editions = con.sql('SELECT is_western, COUNT(*) AS projects, COUNT(*) FILTER (WHERE url LIKE \'%.wikipedia.org\') AS wikipedia_editions FROM sitelink GROUP BY 1').pl()
print(editions)
print('notability filled in v3.2:', con.sql('SELECT COUNT(*) FILTER (WHERE notability.cross_cultural_score IS NOT NULL) FROM individual_enriched').fetchone()[0])
PEOPLE = {'Q9960': 'Ronald Reagan', 'Q8023': 'Nelson Mandela', 'Q2831': 'Michael Jackson', 'Q302': 'Jesus Christ'}
con.sql('''
    SELECT individual_sitelink.qid,
           COUNT(DISTINCT individual_sitelink.site_url) FILTER (WHERE sitelink.is_western) AS western_editions,
           COUNT(DISTINCT individual_sitelink.site_url) FILTER (WHERE NOT sitelink.is_western) AS non_western_editions
    FROM individual_sitelink JOIN sitelink ON sitelink.url = individual_sitelink.site_url
    WHERE list_contains($qids, individual_sitelink.qid)
    GROUP BY 1
''', params={'qids': list(PEOPLE)}).pl().with_columns(pl.col('qid').replace_strict(PEOPLE).alias('name'))

shape: (3, 3)
┌────────────┬──────────┬────────────────────┐
│ is_western ┆ projects ┆ wikipedia_editions │
│ ---        ┆ ---      ┆ ---                │
│ bool       ┆ i64      ┆ i64                │
╞════════════╪══════════╪════════════════════╡
│ false      ┆ 98       ┆ 98                 │
│ true       ┆ 75       ┆ 75                 │
│ null       ┆ 434      ┆ 174                │
└────────────┴──────────┴────────────────────┘
notability filled in v3.2: 0


qid,western_editions,non_western_editions,name
str,i64,i64,str
"""Q302""",74,84,"""Jesus Christ"""
"""Q9960""",75,81,"""Ronald Reagan"""
"""Q2831""",74,84,"""Michael Jackson"""
"""Q8023""",68,87,"""Nelson Mandela"""


## Steps not yet done in v3.2 (placeholders in the text)

In [8]:
print('is_human filled (fictional filter, enrichment 04):', con.sql('SELECT COUNT(is_human) FROM individual').fetchone()[0])
print('is_settlement filled (urban settlements, enrichment 01):', con.sql('SELECT COUNT(is_settlement) FROM place').fetchone()[0])
con.close()

is_human filled (fictional filter, enrichment 04): 0
is_settlement filled (urban settlements, enrichment 01): 0
